<a href="https://colab.research.google.com/github/Jason-fy-wang/LLM_agent/blob/main/huggface_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install torch torchvision transformers peft datasets


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

In [3]:
print(torch.cuda.device_count())


1


In [ ]:
model_name = "Qwen/Qwen3-1.7B"

# 1. load model
tokenizer = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map="auto", dtype=torch.float16)

# 2. construct lora config
lora_config = LoraConfig (
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias = "none",
    target_modules = ["q_pro","v_proj"]
)

# 3. inject lora config
model = get_peft_model(base_model, lora_config)

# 4. construct train data
data_dict = {
    "text": [
        "客服态度好， 处理问题及时",
        "屏幕亮度太低, 看起来很费眼",
        "物流很快, 包装很贴心",
        "使用几天就死机, 非常糟糕"
    ],
    "label": ["正面", "负面", "正面","负面"]
}

dataset = Dataset.from_dict(data_dict)

# 5. pre-process dataset and Prompt construct
def preprocess(sample):
  prompt = f"请判断一下句子的情感倾向, {sample['text']}, answer: {sample['label']}"
  return tokenizer(prompt, truncation=True, padding="max_length", max_length=128)

tokenizered_dataset = dataset.map(preprocess)

# 6: config the train parameter
train_args = TrainingArguments(
    output_dir="./output/qwen_lora_setiment",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    logging_dir= "./logs",
    logging_steps = 5,
    save_strategy="epoch",
    save_total_limit=2,
    fp16 = True,
    learning_rate = 3e-4
)

# 7. construct Trainer instance
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenizered_dataset,
)

# 8. begin train
trainer.train()

# 9. save Lora weight
model.save_pretrained("./output/qwen_lora_setiment")





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
